###  Setup e Configuração
Verifica a existência da variáveis de ambiente, valida a conexão com ADLS e lista os arquivos dispiníveis no container raw/batch-data.

In [0]:
# Instalação de dependências
%pip install python-dotenv azure-storage-file-datalake azure-identity pandas pyarrow pyodbc --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
# Verifica a existência de um arquivo .env
import os
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv()

if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Variáveis de Ambiente encontradas no caminho:{dotenv_path} ")
else:
    print("Variáveis de Ambiente não encontradas.")

In [0]:
# Cria um dicionário com as credenciais
adls_credentials = {
    "CLIENT_ID": os.getenv("CLIENT_ID"),
    "TENANT_ID": os.getenv("TENANT_ID"),
    "CLIENT_SECRET": os.getenv("CLIENT_SECRET"),
    "STORAGE_ACCOUNT_NAME": os.getenv("STORAGE_ACCOUNT_NAME"),
    "CONTAINER_NAME": os.getenv("CONTAINER_NAME", "raw")
}

jdbc_credentials = {
    "JDBC_HOST": os.getenv("JDBC_HOST"),
    "JDBC_DATABASE": os.getenv("JDBC_DATABASE"),
    "JDBC_USERNAME": os.getenv("JDBC_USERNAME"),
    "JDBC_PASSWORD": os.getenv("JDBC_PASSWORD")
}

print("\nStatus das Credenciais ADLS")

# Verifica se todas as credenciais estão definidas ou não
for k, v in adls_credentials.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")

print("\nStatus das Credenciais SQL Server")

for k, v in jdbc_credentials.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")


all_defined  = all(
    list(adls_credentials.values()) +
    list(jdbc_credentials.values())
)

if all_defined :
    print("\nVariáveis de ambiente carregadas com sucesso!")
else:
    print("\nExistem variáveis de ambiente pendentes!")

In [0]:
# Dicionário de Credenciais OAuth para usar com o Spark
adls_options = {
    f"fs.azure.account.auth.type.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": adls_credentials["CLIENT_ID"],
    f"fs.azure.account.oauth2.client.secret.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": adls_credentials["CLIENT_SECRET"],
    f"fs.azure.account.oauth2.client.endpoint.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_credentials["TENANT_ID"]}/oauth2/token"
}

# Define o Caminho para a pasta batch-data do Container
batch_data_path = (
    f"abfss://{adls_credentials["CONTAINER_NAME"]}@{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net/"
    "batch-data"
)

# Lista o conteúdo do Container
files = (
    spark.read
    .format("binaryFile")
    .options(**adls_options)
    .load(batch_data_path)
    .select("path")
)

files.show(truncate=False)